In [3]:
import numpy as np
import pyvista as pv
from utils.tpms_generator import generate_tpms_voxel_grid

def save_voxel_to_vtu(voxel_array, output_filename="output.vtu"):
    """
    将 3D Numpy 体素数组 (ZYX 格式) 导出为 VTU 格式。
    仅保留实体部分 (值 == 1)，剔除孔洞，方便在 Paraview 中直接可视化。
    """
    # 我们的网格 shape 是 (Nz, Ny, Nx)
    nz, ny, nx = voxel_array.shape

    # 1. 创建 PyVista 的 ImageData (规则网格)
    # 注意：网格的"点"比"单元(Cell)"在每个维度上多 1
    grid_pv = pv.ImageData()
    grid_pv.dimensions = np.array([nx + 1, ny + 1, nz + 1])

    # 设置网格的物理尺寸 (这里假设单个体素的边长为 1/nx)
    grid_pv.spacing = (1.0/nx, 1.0/ny, 1.0/nz)

    # 2. 将 Numpy 数组赋予网格的 Cell Data
    # Numpy 默认的 flatten(order="C") 刚好就是 X 变化最快，完美契合 PyVista 的底层内存要求
    grid_pv.cell_data["Solid_Phase"] = voxel_array.flatten(order="C")

    # 3. 提取实体单元 (阈值过滤：只保留 Solid_Phase == 1 的单元)
    # 这一步会将 ImageData 转换为 UnstructuredGrid (即 VTU 格式)
    print("Thresholding solid cells...")
    solid_vtu = grid_pv.threshold(0.5, scalars="Solid_Phase")

    # 4. 保存为 VTU 文件
    solid_vtu.save(output_filename)
    print(f"Successfully saved {solid_vtu.n_cells} solid hexahedral cells to '{output_filename}'!")

if __name__ == "__main__":
    print("Generating Gyroid voxel grid...")
    # 调用你之前写的 utils 生成体素网格
    grid = generate_tpms_voxel_grid(tpms_type='Gyroid',
                                    Nx=1, Ny=1, Nz=1,
                                    resolution=64,
                                    relative_density=0.15,
                                    is_sheet=True)

    print(f"Generated grid shape (Nz, Ny, Nx): {grid.shape}")
    print(f"Volume fraction: {grid.mean():.4f}")

    # 导出为 VTU
    vtu_name = "Gyroid_Sheet_res64.vtu"
    save_voxel_to_vtu(grid, output_filename=vtu_name)

Generating Gyroid voxel grid...
Generated grid shape (Nz, Ny, Nx): (64, 64, 64)
Volume fraction: 0.1500
Thresholding solid cells...
Successfully saved 39323 solid hexahedral cells to 'Gyroid_Sheet_res64.vtu'!
